# Single-catalog performance and interpretation

Evaluate the saved models, then export the complete point-estimate, empirical-scale, reliability and PCA diagnostic suite. zML uses the calibration+test point holdout; sigmaML/rML use the independent final test. Native comparisons use each estimator's residual, while controlled comparisons hold the residual and object mask fixed.

Reusable calculations live in `turbopdz.evaluation.diagnostics`, plotting in `turbopdz.plotting`, and output orchestration in `turbopdz.analysis`. Figures are PDF files in `RUN_DIR/plots/`; tables are CSV files in `RUN_DIR/tables/`. See `docs/workflow.md` and `docs/analysis.md`.


In [ ]:
CONFIG_PATH = "configs/synthetic-optuna.yaml"
RUN_DIR = "runs/synthetic-optuna"
SHOW_PLOTS = True
RUN_IMPORTANCE = True
PERMUTATION_REPEATS = 5
TOP_PCS = 3
N_PDZ_EXAMPLES = 6
N_BINS = 10


In [ ]:
from pathlib import Path
from IPython.display import display
from turbopdz.workflow import run_stage, load_run

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in candidates if (p / "pyproject.toml").exists()), Path.cwd())
config_path = (PROJECT_ROOT / CONFIG_PATH).resolve()
run_dir = (PROJECT_ROOT / RUN_DIR).resolve()
print(f"config={config_path}\nrun={run_dir}")


In [ ]:
result = run_stage("evaluate", config_path, run_dir)
display(result)


## Diagnostic exports

Export shared-axis comparisons, bin/cumulative metrics with counts, native/common-residual correlations and filtering, frozen calibration profiles, scaled-residual shapes, selector checks and PDZ examples. Calibration/development curves are labeled separately from independent final-test results. A fraction within one empirical error scale is descriptive, not guaranteed Gaussian coverage.

`RUN_IMPORTANCE=True` adds grouped and individual-PC permutation diagnostics, signed raw-PDZ loadings, and low/high-score slices. zML importance reports its configured objective and MAE; sigmaML reports calibrated reliability-filtering AUCs and log-error MAE. Panels are ordered independently by importance. Permutation variability is not a bootstrap interval; PCA signs are arbitrary and slice interpretation is not causal.


In [ ]:
from turbopdz.analysis import write_evaluation_diagnostics
outputs = write_evaluation_diagnostics(
    run_dir, show=SHOW_PLOTS, run_importance=RUN_IMPORTANCE,
    repeats=PERMUTATION_REPEATS, top_pcs=TOP_PCS,
    n_examples=N_PDZ_EXAMPLES, n_bins=N_BINS,
)
display(outputs)


## Inspect tables without rerunning models

Open the exported CSVs or rerun only the diagnostic cell above when changing presentation settings. Completed training notebooks must not be run again in the same directory. Every category's `outputs.csv` records saved files and optional outputs skipped because required data or saved training histories are unavailable.


In [ ]:
import pandas as pd
display(pd.read_csv(run_dir / "tables/zml/summary_table.csv"))
if (run_dir / "tables/sigmaml/sigma_bins_native_and_controlled.csv").exists():
    display(pd.read_csv(run_dir / "tables/sigmaml/sigma_bins_native_and_controlled.csv"))
